# Figure 05 -- interaction counts vs N

Loads `results/scaling/interaction_counts.json`, produced by `bench/scaling/interaction_counts.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("scaling/interaction_counts.json")
cfg, recs, fits = art["config"], art["data"]["records"], art["data"]["fits"]
recs = [r for r in recs if "error" not in r]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Left: raw counts. Right: counts per particle, which is where an O(N) result
# either shows up as a flat line or does not.
for ax, per_particle in ((axes[0], False), (axes[1], True)):
    for i, (field, label) in enumerate(
        (("far_pairs", "M2L (far) pairs"), ("near_pairs", "P2P (near) pairs"))
    ):
        sel = sorted((r for r in recs if r.get(field)), key=lambda r: r["n"])
        if not sel:
            continue
        ys = [
            (r[field] / r["n"]) if per_particle else r[field] for r in sel
        ]
        fit = fits.get(field, {})
        alpha = fit.get("exponent")
        suffix = "" if per_particle or not alpha else f"  ($\\alpha={alpha:.2f}$)"
        ax.plot(
            [r["n"] for r in sel], ys,
            marker=style.MARKERS[i % len(style.MARKERS)],
            color=style.CATEGORICAL[i],
            label=label + suffix,
            markersize=3.4,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("$N$")
    ax.set_ylabel("interactions per particle" if per_particle else "interaction count")
    style.finish(ax, legend_kwargs={"loc": "best", "fontsize": 6.4})

style.footer(
    fig,
    jsonio.config_caption(cfg, ["order", "theta", "basis", "leaf_size", "device", "seed"])
    + f"   exponent fitted for N >= {cfg.get('fit_min_n')}",
)
fig.tight_layout()
style.save(fig, FIG_DIR / "fig05_interaction_counts.pdf")

for name, fit in sorted(fits.items()):
    print(f"{name:<12s} alpha={fit['exponent']:.3f} R2={fit['r_squared']:.4f} "
          f"n={fit['n_points']} over N={fit.get('fit_min_n')}..{fit.get('fit_max_n')}")


## Caption

Accepted M2L (far) and P2P (near) interaction counts against $N$, with fitted
power-law exponents. These are properties of the tree and the acceptance
criterion, independent of hardware, and so are the cleanest statement of the
method's complexity -- wall-clock (figure 4) additionally mixes in memory
bandwidth and kernel launch overhead. **Right:** the same counts per particle,
where linear scaling appears as a flat line. Counts are read from the solver's
public runtime diagnostics. Values from
`results/scaling/interaction_counts.json`.
